# Soldani - Extensive benchmark across ten datasets

`2_7` answered the question on Adult: the model gets the direct effect wrong by a
third, and the five recap answers survive it. One dataset cannot say whether that
holds in general, and it cannot say whether the picture depends on the particular
structure Adult happens to have.

This notebook runs the same two branches over the ten datasets configured in
`src/dataset_configs.py`. For each one, FairMind computes the reference values,
the model computes its own from the pre-aggregated tables, and both sets of
numbers are turned into a report and scored.

The interesting variation is already visible in the reference values alone: the
mediator amplifies the disparity on six datasets, is negligible on three, and
attenuates it on one. A single dataset gives one answer to the second recap
question; ten give three different ones.

## 1. Setup

In [ ]:
from pathlib import Path
import sys

# Find the root by searching the "src" folder
current = Path.cwd()

while current != current.parent:
    if (current / "src").exists():
        REPO_ROOT = current
        break
    current = current.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

## 2. Imports

The dataset configurations live in `src/dataset_configs.py` rather than in this
notebook, so that the tests can check them without executing anything: a wrong
state name or a binning that leaves too many levels fails deep inside pgmpy,
with a message that does not name the dataset at fault.

In [ ]:
import datetime
import json
import os
import traceback

import pandas as pd

from src.llm import LLM_CONFIGS, call_llm
from src.benchmark_common import build_llm_prompt, compute_discrepancies, run_fairmind
from src.dataset_configs import DATASET_CONFIGS

from src.report_pipeline.prompt_builder import build_prompts
from src.report_pipeline.llm_client import call_llm_report, find_unfilled_placeholders
from src.report_pipeline.validator import score_report
from src.report_pipeline.annotate import annotate_recap_answers

LLAMA_HOST = os.environ.get("LLAMA_HOST", "localhost")
LLAMA_PORT = os.environ.get("LLAMA_PORT", "8080")
LLM_CONFIGS[0]["base_url"] = f"http://{LLAMA_HOST}:{LLAMA_PORT}/v1"

print(f"LLM endpoint configured: http://{LLAMA_HOST}:{LLAMA_PORT}/v1")
print(f"Datasets configured: {len(DATASET_CONFIGS)}")

## 3. Run configuration

`DRY_RUN` exercises every cell without touching the network, so the loop and the
aggregation can be checked locally before asking the cluster for an hour.

`DATASETS` restricts the campaign to a subset. Leaving it empty runs all ten;
naming a few is the way to try the pipeline before committing to the full sweep.

In [ ]:
DRY_RUN = False    # True = no network call, simulated output (NOT a valid result)
DATASETS = []      # empty = all; otherwise a list of dataset_name values

configs = [dict(c) for c in DATASET_CONFIGS
           if not DATASETS or c["dataset_name"] in DATASETS]

print(f"Running on {len(configs)} dataset(s):")
for c in configs:
    print(f"  {c['dataset_name']:24} {c['combinations']:3} (z,w) pairs")

## 4. One dataset, both branches

The function below is the whole experiment for a single dataset. It mirrors
`2_7` step by step, and returns a record rather than printing, so the loop can
aggregate.

Failures are caught and recorded instead of stopping the sweep. A dataset whose
model answer cannot be parsed, or whose computation gets truncated, should not
cost the other nine: the record keeps the traceback so the cause stays visible
in the output notebook.

In [ ]:
def simulate_report(user_prompt):
    """Fill the placeholders locally, for the dry run only."""
    from src.report_pipeline.llm_client import extract_latex_document
    mock = extract_latex_document(user_prompt)
    for ph in ["QUALITATIVE_TOTAL", "QUALITATIVE_DE", "QUALITATIVE_IE"]:
        mock = mock.replace(f"<<{ph}>>", "[SIMULATED TEXT - DRY RUN, not real content]")
    for i in range(1, 6):
        mock = mock.replace(f"<<ANSWER_Q{i}>>", "YES")
    return mock


def run_one(config, report_date, out_dir):
    """Both branches on one dataset. Returns a record, never raises."""
    name = config["dataset_name"]
    record = {"dataset": name, "status": "ok", "error": None}

    try:
        ground_truth, bn, n_rows, fairmind_time = run_fairmind(config)
        record["n_rows"] = n_rows
        record["fairmind_effects"] = ground_truth
        record["fairmind_seconds"] = round(fairmind_time, 4)

        context = {
            "dataset": name,
            "protected_attr": config["protected"],
            "x0": str(config["x0"]),
            "x1": str(config["x1"]),
            "outcome_attr": f"{config['target_col']} ({config['target_val']})",
            "mediator": ", ".join(config["mediators"]),
            "confounder": ", ".join(config["confounders"]),
        }

        # --- branch A: the model interprets exact numbers -------------------
        effects_A = {**ground_truth, "IE": -ground_truth["IE_reverse"]}
        sys_A, usr_A = build_prompts(effects_A, context, report_date)
        if DRY_RUN:
            report_A, usage_A, time_A = simulate_report(usr_A), {"output_tokens": None}, 0.0
        else:
            report_A, usage_A, time_A = call_llm_report(sys_A, usr_A, max_tokens=4096,
                                                        cache_prompt=False)

        # --- branch B: the model computes the effects ------------------------
        prompt_B = build_llm_prompt(config, bn, n_rows)
        if DRY_RUN:
            llm_effects = {k: ground_truth[k] * 1.05 for k in ["TV", "TE", "DE", "IE"]}
            usage_B, time_B = {"output_tokens": None}, 0.0
        else:
            llm_effects, usage_B, time_B = call_llm(prompt_B, max_tokens=16384,
                                                    cache_prompt=False)
        llm_effects["SE"] = llm_effects["TV"] - llm_effects["TE"]
        llm_effects = {k: float(llm_effects[k]) for k in ["TV", "TE", "SE", "DE", "IE"]}
        record["llm_effects"] = llm_effects

        # Both forms derived from the model's own TE and DE.
        ie_additive = llm_effects["TE"] - llm_effects["DE"]
        ie_reverse = llm_effects["DE"] - llm_effects["TE"]

        effects_B = {**llm_effects, "IE": ie_additive}
        sys_B, usr_B = build_prompts(effects_B, context, report_date)
        if DRY_RUN:
            report_B, usage_B2, time_B2 = simulate_report(usr_B), {"output_tokens": None}, 0.0
        else:
            report_B, usage_B2, time_B2 = call_llm_report(sys_B, usr_B, max_tokens=4096,
                                                          cache_prompt=False)

        # --- scoring ---------------------------------------------------------
        llm_for_rules = {**llm_effects, "IE_reverse": ie_reverse}
        scores = {
            "A_vs_fairmind": score_report(report_A, ground_truth),
            "B_vs_fairmind": score_report(report_B, ground_truth),
            "B_vs_llm_own": score_report(report_B, llm_for_rules),
        }
        record["scoring"] = {k: v.to_dict() for k, v in scores.items()}
        record["discrepancies"] = compute_discrepancies(ground_truth,
                                                        llm_effects).to_dict(orient="records")
        record["unfilled"] = {"A": find_unfilled_placeholders(report_A),
                              "B": find_unfilled_placeholders(report_B)}
        record["timing"] = {"A_report": round(time_A, 2),
                            "B_computation": round(time_B, 2),
                            "B_report": round(time_B2, 2)}
        record["output_tokens"] = {"A_report": usage_A.get("output_tokens"),
                                   "B_computation": usage_B.get("output_tokens"),
                                   "B_report": usage_B2.get("output_tokens")}

        for tag, report, score in [("A_fairmind_numbers", report_A, scores["A_vs_fairmind"]),
                                   ("B_llm_numbers", report_B, scores["B_vs_fairmind"])]:
            path = f"{out_dir}/{name}_{tag}.tex"
            with open(path, "w", encoding="utf-8") as f:
                f.write(report)
            with open(path.replace(".tex", "_annotated.tex"), "w", encoding="utf-8") as f:
                f.write(annotate_recap_answers(report, score))

    except Exception as exc:
        record["status"] = "failed"
        record["error"] = f"{type(exc).__name__}: {exc}"
        record["traceback"] = traceback.format_exc()

    return record

## 5. The sweep

Each dataset makes three requests to the model, and the middle one is the long
one. On Adult it took just over two minutes, so a full sweep is roughly half an
hour of inference plus the fitting.

Progress is printed as it goes rather than at the end, so that a job watched
through the Slurm log shows where it has got to.

In [ ]:
REPORT_DATE = datetime.date.today().isoformat()
ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
out_dir = f"benchmark_results/extensive/{ts}"
os.makedirs(out_dir, exist_ok=True)

records = []
for i, config in enumerate(configs, start=1):
    print(f"[{i}/{len(configs)}] {config['dataset_name']} ... ", end="", flush=True)
    record = run_one(config, REPORT_DATE, out_dir)
    records.append(record)
    if record["status"] == "ok":
        s = record["scoring"]
        print(f"ok  A={s['A_vs_fairmind']['score_pct']} "
              f"B={s['B_vs_fairmind']['score_pct']} "
              f"Bown={s['B_vs_llm_own']['score_pct']}  "
              f"({record['timing']['B_computation']}s on the computation)")
    else:
        print(f"FAILED  {record['error']}")

ok = [r for r in records if r["status"] == "ok"]
print(f"\ncompleted: {len(ok)}/{len(records)}")

## 6. Results

Three tables. The first is the one the supervisor asked for: whether a numerical
error reaches the interpretation, dataset by dataset.

In [ ]:
summary = pd.DataFrame([{
    "dataset": r["dataset"],
    "rows": r["n_rows"],
    "A_vs_fairmind": r["scoring"]["A_vs_fairmind"]["score_pct"],
    "B_vs_fairmind": r["scoring"]["B_vs_fairmind"]["score_pct"],
    "B_vs_llm_own": r["scoring"]["B_vs_llm_own"]["score_pct"],
    "DE_error_%": next(d["rel_error_%"] for d in r["discrepancies"] if d["effect"] == "DE"),
} for r in ok])
print(summary.to_string(index=False))

### Where the model's numbers change the expected answer

The two reference dictionaries disagree only when the model's error crosses a
threshold. Those are the cases where a wrong number becomes a wrong answer, and
they are what distinguishes an error that matters from one that does not.

In [ ]:
flips = []
for r in ok:
    a = r["scoring"]["B_vs_fairmind"]["questions"]
    b = r["scoring"]["B_vs_llm_own"]["questions"]
    for x, y in zip(a, b):
        if x["ground_truth"] != y["ground_truth"]:
            flips.append({"dataset": r["dataset"], "question": x["index"],
                          "expected_fairmind": x["ground_truth"],
                          "expected_llm_numbers": y["ground_truth"],
                          "model_answered": x["llm_answer"]})

if flips:
    print(pd.DataFrame(flips).to_string(index=False))
else:
    print("No expected answer flips on any dataset: every numerical error stayed")
    print("inside the thresholds.")

### The reference decomposition, dataset by dataset

Reported because it is what makes the campaign worth running: the role of the
mediator is not the same everywhere, so the second recap question does not have
one answer across the collection.

In [ ]:
def mediator_role(e):
    ie, de = e["IE_reverse"], e["DE"]
    if abs(ie) < 0.005:
        return "negligible"
    return "amplifies" if ie * de < 0 else "attenuates"

decomposition = pd.DataFrame([{
    "dataset": r["dataset"],
    "TV": round(r["fairmind_effects"]["TV"], 4),
    "TE": round(r["fairmind_effects"]["TE"], 4),
    "DE": round(r["fairmind_effects"]["DE"], 4),
    "IE_reverse": round(r["fairmind_effects"]["IE_reverse"], 4),
    "mediator": mediator_role(r["fairmind_effects"]),
} for r in ok])
print(decomposition.to_string(index=False))
print()
print(decomposition["mediator"].value_counts().to_string())

## 7. Saving

In [ ]:
out = {
    "timestamp": ts,
    "dry_run": DRY_RUN,
    "n_datasets": len(records),
    "n_completed": len(ok),
    "records": records,
}
json_path = f"{out_dir}/summary.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(out, f, indent=2, ensure_ascii=False, default=float)

summary.to_csv(f"{out_dir}/summary.csv", index=False)
decomposition.to_csv(f"{out_dir}/decomposition.csv", index=False)

print(f"Results saved under {out_dir}/")
for name in sorted(os.listdir(out_dir))[:6]:
    print(f"  {name}")
print(f"  ... {len(os.listdir(out_dir))} files in total")